In [1]:
# ===============================
# RG Intertwining Test (A3)
# Single-cell JAX implementation
# ===============================

import jax
import jax.numpy as jnp
from jax import grad, vmap, jit

# -------------------------------
# SU(2) exponential / logarithm
# -------------------------------

def su2_exp(x):
    theta = jnp.linalg.norm(x, axis=-1, keepdims=True)
    half = 0.5 * theta
    small = theta < 1e-8
    w = jnp.cos(half)
    xyz = jnp.where(small, 0.5 * x, jnp.sin(half) * x / theta)
    return jnp.concatenate([w, xyz], axis=-1)

def su2_log(q):
    w, v = q[..., :1], q[..., 1:]
    nv = jnp.linalg.norm(v, axis=-1, keepdims=True)
    angle = 2.0 * jnp.arctan2(nv, w)
    small = nv < 1e-8
    return jnp.where(small, 2.0 * v, angle * v / nv)

# -------------------------------
# Quaternion algebra
# -------------------------------

def quat_mul(q1, q2):
    w1,x1,y1,z1 = jnp.split(q1,4,axis=-1)
    w2,x2,y2,z2 = jnp.split(q2,4,axis=-1)
    return jnp.concatenate([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ], axis=-1)

def quat_inv(q):
    return jnp.concatenate([q[..., :1], -q[..., 1:]], axis=-1)

# -------------------------------
# Karcher (geodesic) mean
# -------------------------------

def karcher_mean(quats, iters=3):
    q = quats[0]
    for _ in range(iters):
        logs = su2_log(quat_mul(quat_inv(q), quats))
        delta = jnp.mean(logs, axis=0)
        q = quat_mul(q, su2_exp(delta))
    return q

# -------------------------------
# Block map π(U)
# -------------------------------

def pi_geodesic(U_block):
    q_mean = karcher_mean(U_block)
    return su2_log(q_mean)

# -------------------------------
# Test function F(Y) = <v,Y>
# -------------------------------

def F_linear(Y, v):
    return jnp.dot(v, Y)

# -------------------------------
# R(U,v) computation
# -------------------------------

def compute_R(U_block, v):
    # Flatten U_block for differentiation
    def F_pullback(flatU):
        U = flatU.reshape((-1,4))
        Y = pi_geodesic(U)
        return F_linear(Y, v)

    grad_f = grad(F_pullback)(U_block.reshape(-1))
    grad_f = grad_f.reshape((-1,4))[:,1:]  # drop quaternion scalar part
    num = jnp.sum(grad_f**2)

    Y = pi_geodesic(U_block)
    denom = jnp.sum(v**2)

    return num / denom

# -------------------------------
# Batched test
# -------------------------------

@jit(static_argnums=1)
def batch_R(key, n_samples=100000, eps=0.1):
    keys = jax.random.split(key, n_samples)

    def sample_one(k):
        k1,k2 = jax.random.split(k)
        X = eps * jax.random.normal(k1, (16,3))
        U = su2_exp(X)
        v = jax.random.normal(k2, (3,))
        return compute_R(U, v)

    return vmap(sample_one)(keys)

# -------------------------------
# Example run
# -------------------------------

key = jax.random.PRNGKey(0)
Rvals = batch_R(key, n_samples=50000, eps=0.1)

print("Estimated C_RG (max R):", jnp.max(Rvals))
print("Mean R:", jnp.mean(Rvals))
print("Expected ~1/16 ≈", 1/16)


TypeError: As of JAX v0.7, parameters to jaxpr equations must have __hash__ and __eq__ methods. In a call to primitive random_split, the value of parameter shape was not hashable: (JitTracer<~int32[]>,)